In [1]:
import numpy as np
import struct
from array import array
from os.path  import join

%matplotlib inline
import random
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

In [3]:
# MNist reader from:
# https://www.kaggle.com/code/hojjatk/read-mnist-dataset

class MnistDataloader(object):
    def __init__(self, training_images_filepath,training_labels_filepath,
                 test_images_filepath, test_labels_filepath):
        self.training_images_filepath = training_images_filepath
        self.training_labels_filepath = training_labels_filepath
        self.test_images_filepath = test_images_filepath
        self.test_labels_filepath = test_labels_filepath
    
    def read_images_labels(self, images_filepath, labels_filepath):        
        labels = []
        with open(labels_filepath, 'rb') as file:
            magic, size = struct.unpack(">II", file.read(8))
            if magic != 2049:
                raise ValueError('Magic number mismatch, expected 2049, got {}'.format(magic))
            labels = array("B", file.read())        
        
        with open(images_filepath, 'rb') as file:
            magic, size, rows, cols = struct.unpack(">IIII", file.read(16))
            if magic != 2051:
                raise ValueError('Magic number mismatch, expected 2051, got {}'.format(magic))
            image_data = array("B", file.read())        
        images = []
        for i in range(size):
            images.append(np.zeros((rows, cols)))
        for i in range(size):
            img = np.array(image_data[i * rows * cols:(i + 1) * rows * cols])
            img = img.reshape(28, 28)
            images[i][:] = img            
        
        return images, labels
            
    def load_data(self):
        x_train, y_train = self.read_images_labels(self.training_images_filepath, self.training_labels_filepath)
        x_test, y_test = self.read_images_labels(self.test_images_filepath, self.test_labels_filepath)
        return (x_train, y_train),(x_test, y_test)   

input_path = 'mnist'
training_images_filepath = join(input_path, 'train-images-idx3-ubyte/train-images-idx3-ubyte')
training_labels_filepath = join(input_path, 'train-labels-idx1-ubyte/train-labels-idx1-ubyte')
test_images_filepath = join(input_path, 't10k-images-idx3-ubyte/t10k-images-idx3-ubyte')
test_labels_filepath = join(input_path, 't10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte')

def show_images(images, title_texts):
    cols = 5
    rows = int(len(images)/cols) + 1
    plt.figure(figsize=(30,20))
    index = 1    
    for x in zip(images, title_texts):        
        image = x[0]        
        title_text = x[1]
        plt.subplot(rows, cols, index)        
        plt.imshow(image, cmap=plt.cm.gray)
        if (title_text != ''):
            plt.title(title_text, fontsize = 15);        
        index += 1
        plt.show

#
# Load MINST dataset
#
mnist_dataloader = MnistDataloader(training_images_filepath, training_labels_filepath, test_images_filepath, test_labels_filepath)
(x_train, y_train), (x_test, y_test) = mnist_dataloader.load_data()

#
# Show some random training and test images 
#
images_to_show = []
titles_to_show = []
for i in range(0, 10):
    r = random.randint(1, 60000)
    images_to_show.append(x_train[r])
    titles_to_show.append('training image [' + str(r) + '] = ' + str(y_train[r]))    

for i in range(0, 5):
    r = random.randint(1, 10000)
    images_to_show.append(x_test[r])        
    titles_to_show.append('test image [' + str(r) + '] = ' + str(y_test[r]))    

#show_images(images_to_show, titles_to_show)

#plt.show()

# end of code from
# https://www.kaggle.com/code/hojjatk/read-mnist-dataset

In [4]:
class MNistDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return torch.tensor(self.x[i], dtype=torch.float32), self.y[i]

dataset = MNistDataset(x_train, y_train)
test_dataset = MNistDataset(x_test, y_test)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=True,
)

In [5]:
class ConvModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=20, kernel_size=(3, 3), padding="same")
        self.bn1 = nn.BatchNorm2d(20)
        self.conv2 = nn.Conv2d(in_channels=20, out_channels=20, kernel_size=(3, 3), padding="same")
        self.bn2 = nn.BatchNorm2d(20)
        self.conv3 = nn.Conv2d(in_channels=20, out_channels=20, kernel_size=(3, 3), padding="same")
        self.bn3 = nn.BatchNorm2d(20)

        self.drop1 = nn.Dropout(0.5)

        self.conv4 = nn.Conv2d(in_channels=20, out_channels=20, kernel_size=(3, 3), padding="same")
        self.bn4 = nn.BatchNorm2d(20)
        self.conv5 = nn.Conv2d(in_channels=20, out_channels=20, kernel_size=(3, 3), padding="same")
        self.bn5 = nn.BatchNorm2d(20)

        self.drop2 = nn.Dropout(0.5)

        self.conv6 = nn.Conv2d(in_channels=20, out_channels=20, kernel_size=(3, 3), padding="same")
        self.bn6 = nn.BatchNorm2d(20)
        self.conv7 = nn.Conv2d(in_channels=20, out_channels=20, kernel_size=(3, 3), padding="same")
        self.bn7 = nn.BatchNorm2d(20)
        self.conv8 = nn.Conv2d(in_channels=20, out_channels=20, kernel_size=(3, 3), padding="same")
        self.bn8 = nn.BatchNorm2d(20)
        
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.lin1 = nn.Linear(20, 20)
        self.lin2 = nn.Linear(20, 10)

    def forward(self, x):
        # normalize
        x = (x / 128) - 0.5
        x.unsqueeze_(-3)
        # (1, 28, 28)
        x = self.conv1(x)
        # (20, 28, 28)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn1(x)

        x = self.conv2(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn2(x)

        x = self.conv3(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn3(x)

        x = self.drop1(x)

        x = self.conv4(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn4(x)
        
        x = self.conv5(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn5(x)

        x = self.drop2(x)
        
        x = self.conv6(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn6(x)

        
        x = self.conv7(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn7(x)

        x = self.conv8(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn8(x)

        x = self.pool(x)
        x = x.reshape((-1, 20, ))
        x = self.lin1(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.lin2(x)

        return x
        


In [44]:
# custom optimizer
# that only uses the sign (+ or -) of the gradient, not its value
class GradSignOptimizer:
    def __init__(self, named_params, *, lr):
        self.named_params = [item for item in named_params]

        # initialize counts
        self.grad_counts = {}
        for n, p in self.named_params:
            self.grad_counts[n] = torch.randint(-64, 64, p.shape, dtype=torch.int8)
        

    def step(self):
        for n, p in self.named_params:
            new_count = 2 * (p.grad > 0) - 1
            self.grad_counts[n] -= (self.grad_counts[n] + 4) // 8
            self.grad_counts[n] += 8 * new_count     # max val will be +- 64 or so

            p.data -= lr * (self.grad_counts[n] / 64.0)

    def zero_grad(self):
        for n, p in self.named_params:
            p.grad = None


In [78]:
torch.manual_seed(0)
model = ConvModel()
model_hooks = []

In [79]:
# how many params
sum([p.numel() for p in model.parameters()])

26490

In [80]:
# check that model runs and shapes work
model(torch.tensor(x_train[2], dtype=torch.float32).unsqueeze(0))

tensor([[-0.0576,  0.2054,  0.0062, -0.0045, -0.0422, -0.1084,  0.0275,  0.1891,
          0.1628,  0.2212]], grad_fn=<AddmmBackward0>)

In [81]:
# training loop

n_batches = 1000
cross_entropy = nn.CrossEntropyLoss()
lr = 0.001


optim = GradSignOptimizer(model.named_parameters(), lr=lr)

# not doing weight decay for now
# l2_lambda = 0
# decay = []
# for p in model.parameters():
#     if p.requires_grad and p.dim() > 1:
#         decay.append(p)

def test_loss():
    model.eval()
    x, y = test_loader.__iter__().__next__()
    with torch.no_grad():
        y_pred = model(x)
    loss = cross_entropy(y_pred, y)
    model.train()
    return loss.item()

model.train()
for i in range(n_batches):
    x, y = loader.__iter__().__next__()
    y_pred = model(x)
    loss = cross_entropy(y_pred, y)
    if i % 50 == 0 or i == n_batches - 1:
        print(f"Loss at step {i:4} is {loss.item():0.3f}")
        print(f"Test loss at step {i:4} is {test_loss():0.3f}")

    # for p in decay:
    #     loss += l2_lambda * (p*p).mean()

    optim.zero_grad()
    loss.backward()
    optim.step()


Loss at step    0 is 2.314
Test loss at step    0 is 2.293
Loss at step   50 is 2.084
Test loss at step   50 is 2.060
Loss at step  100 is 2.071
Test loss at step  100 is 2.018
Loss at step  150 is 1.847
Test loss at step  150 is 1.832
Loss at step  200 is 1.594
Test loss at step  200 is 1.541
Loss at step  250 is 1.372
Test loss at step  250 is 1.282
Loss at step  300 is 1.093
Test loss at step  300 is 1.285
Loss at step  350 is 1.062
Test loss at step  350 is 1.289
Loss at step  400 is 1.180
Test loss at step  400 is 0.933
Loss at step  450 is 1.186
Test loss at step  450 is 1.509
Loss at step  500 is 0.829
Test loss at step  500 is 0.962
Loss at step  550 is 0.635
Test loss at step  550 is 0.668
Loss at step  600 is 0.608
Test loss at step  600 is 0.788
Loss at step  650 is 0.683
Test loss at step  650 is 0.864
Loss at step  700 is 0.718
Test loss at step  700 is 0.570
Loss at step  750 is 0.494
Test loss at step  750 is 0.412
Loss at step  800 is 0.929
Test loss at step  800 is 0.3

In [65]:
optim.grad_counts["conv4.weight"]

tensor([[[[ -1,  -9,  -2],
          [ -1,   5,  -2],
          [ -7,   2,  -9]],

         [[-24, -20, -29],
          [ -9,  -3,   2],
          [-27, -12, -26]],

         [[  2,   7,   2],
          [  0,   2,  20],
          [ 20,  22,  29]],

         ...,

         [[ 12,   8,   4],
          [ -9,   4,   5],
          [ -9,  -7,  12]],

         [[-21,  -8,  -6],
          [-23,  -7,  -4],
          [-20,  -5,  -5]],

         [[ 16, -10,  -3],
          [  8,  30,  16],
          [ 25,  21,  28]]],


        [[[ 31,  13,  -5],
          [ 11, -13, -31],
          [ -5, -32, -45]],

         [[  2, -12,  -5],
          [ 17,   7,  -3],
          [ -5,   8, -18]],

         [[-23,   7,  25],
          [  7,  10,  20],
          [-13, -10,  -5]],

         ...,

         [[-12, -16, -38],
          [-26, -40, -40],
          [-45, -44, -35]],

         [[-10,  -8,  11],
          [ -2,  -3,  11],
          [ -9, -12,   2]],

         [[-35, -20, -24],
          [-34, -35, -27],
 

In [82]:
cml_loss = 0
count = 0

model.eval()
for idx in range(200, 215):
    x, y = test_dataset[idx]
    #show_images([x], [y])
    #plt.show()

    res = model(x.unsqueeze(0))
    #print(res)
    loss = cross_entropy(res, torch.tensor([y])).item()
    print(f"{idx:5} -- {loss:0.2f}")
    cml_loss += loss
    count += 1

print(f"Average: {cml_loss/count:0.2f}")




  200 -- 0.39
  201 -- 0.01
  202 -- 0.00
  203 -- 0.00
  204 -- 0.00
  205 -- 0.08
  206 -- 0.90
  207 -- 0.59
  208 -- 0.27
  209 -- 1.27
  210 -- 0.02
  211 -- 0.32
  212 -- 0.77
  213 -- 0.08
  214 -- 0.25
Average: 0.33
